[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/celoy/intro-ai-ed353/blob/main/notebooks/03b_svm.ipynb)

# Support vector machines

In this notebook we implement a linear support vector machine (SVM) with NumPy, compare it with logistic
regression, and then use the SVMs of scikit-learn to classify data that no straight line can separate,
with a Gaussian kernel.

The data are measurements of adult penguins of three species (Adélie, Chinstrap, Gentoo) nesting in the
Palmer Archipelago (Antarctica), collected by K. B. Gorman and the Palmer Station Long Term Ecological Research
program: Gorman, Williams & Fraser (2014), *PLoS ONE* 9, e90081,
[doi:10.1371/journal.pone.0090081](https://doi.org/10.1371/journal.pone.0090081). We use the file distributed by
the R package `palmerpenguins` (A. M. Horst, A. P. Hill, K. B. Gorman,
[doi:10.5281/zenodo.3960218](https://doi.org/10.5281/zenodo.3960218)), released under the CC0 licence.

Notation: as in the lecture, the labels are now $y = \pm 1$, and the classifier is the sign of
$f(x) = a^T x + b$.

In [ ]:
# Run this cell first. On Google Colab it downloads the course files
# (data, helper modules); on your own computer it does nothing.
import os, sys
if "google.colab" in sys.modules:
    repo = "/content/intro-ai-ed353"
    if not os.path.isdir(repo):
        os.system(f"git clone --depth 1 -q https://github.com/celoy/intro-ai-ed353.git {repo}")
    os.chdir(f"{repo}/notebooks")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
def standardize(X):
    """Standardize each column of X: subtract its mean and divide by its standard deviation."""
    mean_x = X.mean(axis=0)
    std_x = X.std(axis=0)
    return (X - mean_x) / std_x, mean_x, std_x


def plot_svm(x, y, decision, ax=None, support=None, labels=('$y = -1$', '$y = +1$'),
             xlabel='$x_1$', ylabel='$x_2$', title=None):
    """Data, decision regions and margins of a classifier with decision function f.

    decision(points) must return f at the rows of the array points, of shape (n, 2).
    The line f = 0 is the decision boundary; the dashed lines f = -1 and f = +1 are the margins.
    The points listed in support (an array of shape (n_s, 2)) are circled.
    """
    if ax is None:
        ax = plt.gca()
    pad = 0.5
    g0 = np.linspace(x[:, 0].min() - pad, x[:, 0].max() + pad, 200)
    g1 = np.linspace(x[:, 1].min() - pad, x[:, 1].max() + pad, 200)
    G0, G1 = np.meshgrid(g0, g1)
    F = decision(np.column_stack([G0.ravel(), G1.ravel()])).reshape(G0.shape)
    ax.contourf(G0, G1, (F > 0).astype(float), levels=[-0.5, 0.5, 1.5],
                colors=['tab:blue', 'tab:red'], alpha=0.15)
    ax.contour(G0, G1, F, levels=[-1, 0, 1], colors='k', linestyles=['--', '-', '--'], linewidths=1)
    ax.scatter(*x[y < 0].T, marker='.', color='tab:blue', label=labels[0])
    ax.scatter(*x[y > 0].T, marker='*', color='tab:red', label=labels[1])
    if support is not None:
        ax.scatter(*support.T, s=80, facecolors='none', edgecolors='k', label='support vectors')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    if title is not None:
        ax.set_title(title)
    ax.legend(fontsize='small')


df = pd.read_csv('Data/penguins.csv').dropna()   # keep the penguins with complete records
df.head()

## Part 1: a linear SVM in NumPy

We minimize the soft-margin cost of the lecture,
$$J(a, b) = \frac{\lambda}{2}\|a\|^2 + \frac{1}{N}\sum_{i=1}^{N}\max\Big(0,\; 1 - y^{(i)}\big(a^T x^{(i)} + b\big)\Big).$$
The second term (the *hinge loss*) vanishes for the points beyond the margin, $y^{(i)}(a^Tx^{(i)} + b) \ge 1$;
the first term favours a large margin $1/\|a\|$. When the data are linearly separable and $\lambda \to 0$, the
minimizer tends to the maximum-margin (hard-margin) classifier.

### Question 1

Keep the Adélie and Gentoo penguins. Build the standardized feature
matrix `x` (bill length, bill depth) and the labels `y`: $+1$ for Gentoo, $-1$ for Adélie. Plot the data.

In [ ]:
pair = ...
X_raw = ...
y = ...      # +1 for Gentoo, -1 for Adelie

x, mean_x, std_x = standardize(X_raw)
N = len(y)
print(f'N = {N}')

In [ ]:
species_labels = ('Adelie ($y = -1$)', 'Gentoo ($y = +1$)')
plt.scatter(*x[y < 0].T, marker='.', color='tab:blue', label=species_labels[0])
plt.scatter(*x[y > 0].T, marker='*', color='tab:red', label=species_labels[1])
plt.xlabel('bill length (standardized)')
plt.ylabel('bill depth (standardized)')
plt.legend()
plt.show()

### Question 2

Write the function `svm_cost(a, b, x, y, lam)` that returns $J(a, b)$.

In [ ]:
def svm_cost(a, b, x, y, lam):
    """Soft-margin SVM cost: lam / 2 |a|^2 + mean hinge loss."""

    return ...

### Question 3

The hinge loss is not differentiable where $y^{(i)}(a^Tx^{(i)} + b) = 1$, but we can use a *subgradient*: the
gradient of $1 - y^{(i)}(a^Tx^{(i)} + b)$ for the points with $y^{(i)}(a^Tx^{(i)} + b) < 1$, and zero for the
others. This gives
$$g_a = \lambda a - \frac{1}{N}\sum_{i:\, y^{(i)}(a^Tx^{(i)}+b) < 1} y^{(i)}x^{(i)},
\qquad
g_b = - \frac{1}{N}\sum_{i:\, y^{(i)}(a^Tx^{(i)}+b) < 1} y^{(i)}.$$
Write the function `svm_subgradient(a, b, x, y, lam)` that returns $(g_a, g_b)$.

In [ ]:
def svm_subgradient(a, b, x, y, lam):
    """Subgradient (g_a, g_b) of the soft-margin SVM cost."""

    return g_a, g_b

### Question 4

Complete the function `svm_subgradient_descent(x, y, lam)`. Starting from $a = 0$, $b = 0$, it does `n_iter`
steps $a := a - \eta_t g_a$, $b := b - \eta_t g_b$ with the decreasing step $\eta_t = 1/(\lambda t)$ (capped at
1), $t = 1, 2, \dots$ A subgradient step does not always decrease $J$, so the function keeps the best
$(a, b)$ met along the way. It returns $a$, $b$ and the list of costs.

In [ ]:
def svm_subgradient_descent(x, y, lam, n_iter=20000):
    """Minimize the soft-margin SVM cost by subgradient descent; return the best (a, b) and the costs."""
    a = np.zeros(x.shape[1])
    b = 0.0
    a_best, b_best, cost_best = a, b, np.inf
    costs = []
    for t in range(1, n_iter + 1):
        # subgradient step with eta = min(1, 1 / (lam t))

        # cost after the step; keep the best (a, b)

    return a_best, b_best, costs

### Question 5

For $\lambda = 10^{-4}$, $10^{-2}$ and $10^{-1}$: fit the SVM, find the support vectors (the points with
$y^{(i)}(a^Tx^{(i)} + b) \le 1$; use a tolerance of $10^{-2}$ to allow for the finite accuracy of the
minimization), and plot the boundary and the margins with `plot_svm`. Print the margin $1/\|a\|$, the number of
support vectors and the training accuracy. How do they change with $\lambda$?

In [ ]:
lambdas = [1e-4, 1e-2, 1e-1]
fig, axs = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, lam in zip(axs, lambdas):
    a, b, costs = svm_subgradient_descent(x, y, lam)
    support = ...    # boolean array: True for the support vectors
    accuracy = ...
    plot_svm(x, y, lambda p: p @ a + b, ax=ax, support=x[support], labels=species_labels,
             xlabel='bill length (standardized)', ylabel='bill depth (standardized)',
             title=f'$\\lambda$ = {lam:g}')
    print(f'lambda = {lam:g}: J = {costs[-1]:.4f}, margin = {1 / np.linalg.norm(a):.3f}, '
          f'{support.sum()} support vectors, accuracy = {accuracy:.3f}')
plt.show()

### Question 6: check with scikit-learn

From now on we use scikit-learn, the standard Python library for classical machine learning. Its models all
work the same way: create the model with its parameters, fit it with `model.fit(X, y)`, then use
`model.predict(X)`, `model.score(X, y)` (the accuracy for a classifier) or, for an SVM,
`model.decision_function(X)`, which returns $f(x)$.

`sklearn.svm.SVC(kernel='linear', C=C)` minimizes $\frac{1}{2}\|a\|^2 + C\sum_i \max(0, 1 - y^{(i)}(a^Tx^{(i)} + b))$.
Dividing our cost by $\lambda$ shows that the two problems have the same solution for $C = 1/(\lambda N)$.
For $\lambda = 10^{-2}$, fit `SVC` and compare its $a$ (attribute `coef_`), $b$ (`intercept_`) and number of
support vectors (`n_support_`, per class) with your own results.

In [ ]:
from sklearn.svm import SVC

lam = 1e-2
a, b, costs = svm_subgradient_descent(x, y, lam)
clf = ...
print('NumPy       : a =', a, ' b =', b)
print('scikit-learn: a =', clf.coef_[0], ' b =', clf.intercept_[0], ' support vectors:', clf.n_support_.sum())

## Part 2: SVM and logistic regression

With labels $\tilde y = 2y - 1 = \pm 1$ instead of $y \in \{0, 1\}$ and $f(x) = \theta^Tx$, the logistic loss of
one point becomes
$$-\Big[y\log h(f) + (1-y)\log\big(1-h(f)\big)\Big] = \log\big(1 + e^{-\tilde y f}\big),$$
a function of the product $m = \tilde y f$ only, like the hinge loss $\max(0, 1 - m)$. Penalized logistic
regression and the soft-margin SVM thus minimize the same kind of cost, a penalty on the parameters plus the
average of a loss of $m$; only the loss differs.

### Question 7

On the same graph, plot as functions of $m \in [-3, 3]$: the 0–1 loss (1 if $m < 0$, 0 otherwise), which
counts the errors; the hinge loss; and the logistic loss divided by $\log 2$ (so that it equals 1 at $m = 0$).
Which points contribute to each loss?

In [ ]:
m = np.linspace(-3, 3, 601)


### Question 8

Back to the sex of Adélie penguins (bill depth and body mass, as in notebook 03a), with $y = +1$ for a female
and $-1$ for a male: the classes overlap. Fit the SVM with $\lambda = 10^{-2}$, and the penalized logistic regression
of 03a (implemented below) with the same weight on the penalty. Plot both boundaries on the same graph, with the
support vectors of the SVM. Compare the boundaries, the accuracies and the number of points that determine
each boundary.

In [ ]:
def logistic_regression(x, y, lam, alpha=1.0, n_iter=20000):
    """Logistic regression with labels y = -1 or +1 and the cost lam / 2 |a|^2 + mean log(1 + exp(-y f)),
    f = a.x + b, minimized by gradient descent (same penalty as the SVM cost; b is not penalized)."""
    a = np.zeros(x.shape[1])
    b = 0.0
    for _ in range(n_iter):
        m = y * (x @ a + b)
        w = -y / (1 + np.exp(m)) / len(y)      # derivative of the mean loss with respect to f
        a = a - alpha * (lam * a + x.T @ w)
        b = b - alpha * w.sum()
    return a, b


adelie = df[df['species'] == 'Adelie']
x_sex, _, _ = standardize(adelie[['bill_depth_mm', 'body_mass_g']].to_numpy())
y_sex = np.where(adelie['sex'] == 'female', 1.0, -1.0)
sex_labels = ('male ($y = -1$)', 'female ($y = +1$)')

In [ ]:
lam = 1e-2
a_svm, b_svm, _ = ...
a_log, b_log = ...
support = ...

plot_svm(x_sex, y_sex, lambda p: p @ a_svm + b_svm, support=x_sex[support], labels=sex_labels,
         xlabel='bill depth (standardized)', ylabel='body mass (standardized)')
# add the logistic boundary f = 0 as a line


## Part 3: kernels

The solution of the SVM can be written $f(x) = \sum_i \alpha_i y^{(i)}\, x^{(i)T}x + b$, where $\alpha_i \neq 0$
only for the support vectors, and the dual problem that gives the $\alpha_i$ also involves the data only through
the scalar products $x^{(i)T}x^{(j)}$. Replacing every scalar product by a kernel $K(x^{(i)}, x)$ amounts to
using a linear SVM on transformed features, without ever computing them. With the Gaussian kernel
$$K(x, x') = \exp\big(-\gamma\|x - x'\|^2\big),$$
$f(x) = \sum_i \alpha_i y^{(i)} K(x^{(i)}, x) + b$ is a sum of Gaussian bumps of width $\gamma^{-1/2}$ centred
on the support vectors. In scikit-learn: `SVC(kernel='rbf', gamma=gamma, C=C)`.

### Question 9

Chinstrap penguins have long and deep bills: they occupy a corner of the (bill length, bill depth) plane,
with Adélie penguins on one side and Gentoo penguins on the other. We try to tell Chinstrap penguins ($y = +1$)
from the two other species ($y = -1$). The cell below splits the data into a training set (70 %) and a test
set (30 %), and standardizes both with the mean and standard deviation of the training set. Fit a linear SVM
(`SVC(kernel='linear', C=1)`) on the training set, plot it, and print its accuracy on both sets. Compare with
the accuracy of the classifier that always answers "not a Chinstrap".

In [ ]:
X3 = df[['bill_length_mm', 'bill_depth_mm']].to_numpy()
y3 = np.where(df['species'] == 'Chinstrap', 1.0, -1.0)

rng = np.random.default_rng(0)
perm = rng.permutation(len(y3))
n_train = int(0.7 * len(y3))
train, test = perm[:n_train], perm[n_train:]
x_train, mean3, std3 = standardize(X3[train])
x_test = (X3[test] - mean3) / std3
y_train, y_test = y3[train], y3[test]
chinstrap_labels = ('Adelie or Gentoo ($y = -1$)', 'Chinstrap ($y = +1$)')
axis_labels = dict(xlabel='bill length (standardized)', ylabel='bill depth (standardized)')

In [ ]:
clf = ...

plot_svm(x_train, y_train, clf.decision_function, support=clf.support_vectors_,
         labels=chinstrap_labels, **axis_labels)
plt.show()


### Question 10

Fit SVMs with a Gaussian kernel, `C=1` and $\gamma = 0.1, 1, 10, 100$. For each, plot the decision regions
(training set) and print the number of support vectors and the training and test accuracies. Then fix
$\gamma = 1$ and try $C = 0.01, 1, 100$. How do $\gamma$ and $C$ control the complexity of the boundary? Which
values would you choose, and how?

In [ ]:
fig, axs = plt.subplots(1, 4, figsize=(20, 4.5))
for ax, gamma in zip(axs, [0.1, 1, 10, 100]):
    clf = ...
